In [1]:
# Install these only if your environment does not already have them.
%pip install -q datasets transformers evaluate jiwer accelerate soundfile librosa matplotlib seaborn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.5 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import io
import re
from collections import Counter
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from datasets import Audio, load_dataset
from IPython.display import Audio as IPythonAudio, display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)

print("Imports ready")


Imports ready


In [3]:
# Adjust this if you want another locale from MINDS-14.
DATASET_NAME = "PolyAI/minds14"
DATASET_CONFIG = "en-US"  # change to the locale you are using
TARGET_SAMPLING_RATE = 16000
RANDOM_SEED = 42

# Download (if not cached) or load MINDS-14 from Hugging Face Hub.
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
raw_dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-US/train-00000-of-00001.parquet:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})

In [4]:
for split_name, split_ds in raw_dataset.items():
    print(f"Split: {split_name}")
    print(f"  Rows: {len(split_ds)}")
    print(f"  Columns: {split_ds.column_names}")
    print(f"  Features: {split_ds.features}")
    print()

Split: train
  Rows: 563
  Columns: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id']
  Features: {'path': Value('string'), 'audio': Audio(sampling_rate=8000, decode=True, stream_index=None), 'transcription': Value('string'), 'english_transcription': Value('string'), 'intent_class': ClassLabel(names=['abroad', 'address', 'app_error', 'atm_limit', 'balance', 'business_loan', 'card_issues', 'cash_deposit', 'direct_debit', 'freeze', 'high_value_payment', 'joint_account', 'latest_transactions', 'pay_bill']), 'lang_id': ClassLabel(names=['cs-CZ', 'de-DE', 'en-AU', 'en-GB', 'en-US', 'es-ES', 'fr-FR', 'it-IT', 'ko-KR', 'nl-NL', 'pl-PL', 'pt-PT', 'ru-RU', 'zh-CN'])}

